# 04. Проверка синтетики: насколько она похожа на реальные описания

Сравниваем набор синтетики с реальными описаниями пилота (100 карточек). Val и test не читаем.
Смотрим четыре вещи: длину, какие атрибуты названы, как записаны значения, насколько тексты однообразны.
Имя набора меняется в первой ячейке (`NAME`).

In [ ]:
import json, re, random
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

NAME = "fewshot"      # какой набор проверяем: fewshot, heldout, train
SYN = f"../data/synthetic/{NAME}.jsonl"
EDA = "../03_eda/outputs/dns_cards_500"

syn = [json.loads(l) for l in open(SYN, encoding="utf-8")]
pilot_ids = set(json.load(open(f"{EDA}/split_ids.json"))["pilot"])
real = [c["description"] for c in json.load(open("../data/dns_cards_500.json", encoding="utf-8")) if c["url"] in pilot_ids]
syn_text = [r["description"] for r in syn]
print("синтетика:", len(syn_text), "| реальные (пилот):", len(real))

## 1. Длина описания
Реальные описания разной длины. Если синтетика почти одной длины, это заметно.

In [ ]:
L = pd.DataFrame({"синтетика": pd.Series([len(t) for t in syn_text]).describe(),
                  "реальные": pd.Series([len(t) for t in real]).describe()}).round(0)
display(L)
plt.hist([len(t) for t in real], bins=15, alpha=0.6, label="реальные")
plt.hist([len(t) for t in syn_text], bins=15, alpha=0.6, label="синтетика")
plt.xlabel("символов"); plt.legend(); plt.show()

## 2. Какие атрибуты названы
Доля описаний, где атрибут назван в тексте (эталон не `null`). У пилота берём из `gold_pairs.csv`.

In [ ]:
ATTRS = ["диагональ", "разрешение", "частота", "тип матрицы", "яркость", "контрастность", "изогнутость"]
pairs = pd.read_csv(f"{EDA}/gold_pairs.csv")
pairs = pairs[pairs["url"].isin(pilot_ids)]
share = pd.DataFrame({
    "синтетика": {a: sum(r["gold"][a] is not None for r in syn) / len(syn) for a in ATTRS},
    "реальные": {a: pairs[pairs["атрибут"] == a]["эталон"].notna().mean() for a in ATTRS}}).round(2)
display(share)
print("названо атрибутов на описание, в среднем (синтетика):", round(sum(v is not None for r in syn for v in r["gold"].values()) / len(syn), 2))

## 3. Как записаны значения
Считаем по текстам, сколько описаний используют каждую запись. Это те же форматы, что в анализе 03.

In [ ]:
FORMS = {
    "диагональ: 27\"":          r'\d\s*(?:"|”|″)',
    "диагональ: 27-дюймовый":   r"\d\s*-\s*дюйм",
    "диагональ: 27 дюймов":     r"\d\s+дюйм",
    "дробь через точку":        r"\d\.\d+\s*(?:\"|”|″|-?\s*дюйм)",
    "дробь через запятую":      r"\d,\d+\s*(?:\"|”|″|-?\s*дюйм)",
    "разрешение: x (латиница)": r"\d\s*x\s*\d{3,4}",
    "разрешение: х (кириллица)": r"\d\s*х\s*\d{3,4}",
    "разрешение: ×":            r"\d\s*×\s*\d{3,4}",
    "частота: Гц":              r"\d\s*гц",
    "частота: герц":            r"\d\s*герц",
    "яркость: кд/м²":           r"\d\s*кд",
    "яркость: нит":             r"\d\s*нит",
    "яркость: cd/m2":           r"\d\s*cd",
    "контрастность: 3000:1":    r"\d\s*:\s*1\b",
    "контрастность: 3000 к 1":  r"\d\s*к\s*1\b",
}
def count(texts, pattern):
    return sum(bool(re.search(pattern, t.lower())) for t in texts)

F = pd.DataFrame({"синтетика, штук": {k: count(syn_text, p) for k, p in FORMS.items()},
                  "реальные, штук": {k: count(real, p) for k, p in FORMS.items()}})
F["синтетика, %"] = (F["синтетика, штук"] / len(syn_text) * 100).round(0)
F["реальные, %"] = (F["реальные, штук"] / len(real) * 100).round(0)
display(F)

## 4. Однообразие
- Сколько разных начал у описаний (первые три слова).
- Насколько тексты похожи друг на друга: для каждого берём самое похожее другое описание (косинус символьных n-грамм, как в анализе 03). Сравниваем с таким же числом случайных реальных описаний.
- Не списано ли у реальных: самое похожее реальное описание для каждого синтетического. Близость от 0,9 считаем почти копией.

In [ ]:
starts = pd.Series([" ".join(t.split()[:3]) for t in syn_text])
print("разных начал:", starts.nunique(), "из", len(starts))
display(starts.value_counts().head(8))

def max_sim(texts):
    X = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit_transform(texts)
    S = cosine_similarity(X); 
    for i in range(len(texts)): S[i, i] = 0
    return S.max(axis=1)

random.seed(0)
real_sample = random.sample(real, min(len(syn_text), len(real)))
print("самое похожее другое описание, среднее:  синтетика", max_sim(syn_text).mean().round(2),
      "| реальные", max_sim(real_sample).mean().round(2))

X = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(syn_text + real)
cross = cosine_similarity(X.transform(syn_text), X.transform(real)).max(axis=1)
print("похожесть на реальные: среднее", cross.mean().round(2), "| максимум", cross.max().round(2), "| почти копий (от 0,9):", int((cross >= 0.9).sum()))

## 5. Как читать результат
- **Длина.** Если у синтетики разброс заметно уже, чем у реальных, длину надо варьировать в промпте.
- **Атрибуты.** Доли должны быть близки к реальным; сильные расхождения ломают сравнение «синтетика → реальные».
- **Записи значений.** Если какая-то запись есть в реальных текстах, а в синтетике нет (или наоборот), модели учатся на другом.
- **Однообразие.** Синтетика заметно более похожа сама на себя, чем реальные, значит, примеры однотипные: нужны другие темы и стили в промпте.
- **Почти копии.** Если они есть, описание списано с образца стиля и его надо убрать.